# 02 - Extracao dos dados de CNPJ - MG

Neste notebook, executo a extracao dos arquivos publicos de CNPJ da Receita Federal para montar uma base bruta consolidada de empresas ativas em Minas Gerais.

A saida desta etapa sera um CSV em `data/raw/cnpj/cnpj_mg_extracao.csv`, que depois podera ser tratado, segmentado e analisado nas proximas etapas do projeto.

## 0. Como organizar os arquivos de entrada

Arquivos esperados para esta etapa:

- Estabelecimentos: arquivos com `Estabelecimentos` no nome.
- Empresas: arquivos com `Empresas` no nome.
- Municipios: arquivo com `Municipios` no nome.
- CNAEs: arquivo com `Cnaes` no nome.

Os arquivos de estabelecimentos e empresas podem vir particionados em varios ZIPs. O notebook le todos os arquivos encontrados na pasta.

In [ ]:
from pathlib import Path
from zipfile import ZipFile

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "cnpj"
DOWNLOAD_DIR = RAW_DIR / "downloads"
OUTPUT_CSV = RAW_DIR / "cnpj_mg_extracao.csv"

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

DOWNLOAD_DIR, OUTPUT_CSV

## 1. Definicao dos layouts da Receita Federal

Os arquivos da Receita Federal nao vem com cabecalho. Por isso, defino manualmente os nomes das colunas que serao usados na leitura dos CSVs.

In [ ]:
ESTABELECIMENTOS_COLUMNS = [
    "cnpj_basico",
    "cnpj_ordem",
    "cnpj_dv",
    "matriz_filial",
    "nome_fantasia",
    "situacao_cadastral",
    "data_situacao_cadastral",
    "motivo_situacao_cadastral",
    "nome_cidade_exterior",
    "pais",
    "data_inicio_atividade",
    "cnae_fiscal_principal",
    "cnae_fiscal_secundaria",
    "tipo_logradouro",
    "logradouro",
    "numero",
    "complemento",
    "bairro",
    "cep",
    "uf",
    "municipio_codigo",
    "ddd_1",
    "telefone_1",
    "ddd_2",
    "telefone_2",
    "ddd_fax",
    "fax",
    "email",
    "situacao_especial",
    "data_situacao_especial",
]

EMPRESAS_COLUMNS = [
    "cnpj_basico",
    "nome_empresarial",
    "natureza_juridica",
    "qualificacao_responsavel",
    "capital_social",
    "porte_empresa",
    "ente_federativo_responsavel",
]

MUNICIPIOS_COLUMNS = ["municipio_codigo", "municipio_nome"]
CNAES_COLUMNS = ["cnae_fiscal_principal", "cnae_fiscal_descricao"]

OUTPUT_COLUMNS = [
    "cnpj_basico",
    "cnpj_ordem",
    "cnpj_dv",
    "cnpj",
    "nome_empresarial",
    "nome_fantasia",
    "matriz_filial",
    "situacao_cadastral",
    "data_situacao_cadastral",
    "motivo_situacao_cadastral",
    "data_inicio_atividade",
    "cnae_fiscal_principal",
    "cnae_fiscal_descricao",
    "cnae_fiscal_secundaria",
    "tipo_logradouro",
    "logradouro",
    "numero",
    "complemento",
    "bairro",
    "cep",
    "uf",
    "municipio_codigo",
    "municipio_nome",
    "ddd_1",
    "telefone_1",
    "ddd_2",
    "telefone_2",
    "email",
    "capital_social",
    "porte_empresa",
    "natureza_juridica",
    "opcao_simples",
    "opcao_mei",
    "fonte_arquivo",
    "data_extracao",
]

len(OUTPUT_COLUMNS)

## 2. Funcoes auxiliares

As funcoes abaixo procuram os ZIPs na pasta de downloads e leem os CSVs internos usando o padrao da Receita Federal: separador `;`, codificacao `latin1` e ausencia de cabecalho.

In [ ]:
def find_zip_files(keyword: str) -> list[Path]:
    keyword = keyword.lower()
    return sorted(path for path in DOWNLOAD_DIR.glob("*.zip") if keyword in path.name.lower())


def first_file_inside_zip(zip_path: Path) -> str:
    with ZipFile(zip_path) as zip_file:
        file_names = [name for name in zip_file.namelist() if not name.endswith("/")]
    if not file_names:
        raise ValueError(f"ZIP sem arquivo interno: {zip_path}")
    return file_names[0]


def read_zip_csv(zip_path: Path, columns: list[str], chunksize: int | None = None):
    inner_file = first_file_inside_zip(zip_path)
    return pd.read_csv(
        zip_path,
        compression="zip",
        header=None,
        names=columns,
        sep=";",
        dtype=str,
        encoding="latin1",
        chunksize=chunksize,
        low_memory=False,
    )


def read_single_table(keyword: str, columns: list[str]) -> pd.DataFrame:
    files = find_zip_files(keyword)
    if not files:
        raise FileNotFoundError(f"Nenhum ZIP encontrado com '{keyword}' em {DOWNLOAD_DIR}")
    return read_zip_csv(files[0], columns)


def build_cnpj(df: pd.DataFrame) -> pd.Series:
    return df["cnpj_basico"] + df["cnpj_ordem"] + df["cnpj_dv"]

## 3. Conferencia dos arquivos disponiveis

Esta celula lista os arquivos encontrados antes da extracao. Se alguma lista vier vazia, baixe o ZIP correspondente antes de continuar.

In [ ]:
available_files = {
    "estabelecimentos": find_zip_files("estabele"),
    "empresas": find_zip_files("empresa"),
    "municipios": find_zip_files("municip"),
    "cnaes": find_zip_files("cnae"),
}

for source_name, files in available_files.items():
    print(f"{source_name}: {len(files)} arquivo(s)")
    for file in files[:5]:
        print(f"  - {file.name}")

## 4. Leitura das tabelas de apoio

Municipios e CNAEs sao tabelas menores. Elas entram como apoio para trocar codigos por nomes e descricoes.

In [ ]:
municipios = read_single_table("municip", MUNICIPIOS_COLUMNS)
cnaes = read_single_table("cnae", CNAES_COLUMNS)

municipios.head(), cnaes.head()

## 5. Extracao dos estabelecimentos ativos em MG

A tabela de estabelecimentos costuma ser a maior da base. Por isso, a leitura e feita em partes (`chunks`) e ja filtra apenas registros de Minas Gerais com situacao cadastral ativa (`02`).

In [ ]:
CHUNKSIZE = 500_000

estabelecimentos_files = find_zip_files("estabele")

filtered_chunks = []
total_rows = 0
total_selected = 0

for zip_path in estabelecimentos_files:
    print(f"Lendo {zip_path.name}...")
    for chunk in read_zip_csv(zip_path, ESTABELECIMENTOS_COLUMNS, chunksize=CHUNKSIZE):
        total_rows += len(chunk)
        selected = chunk[(chunk["uf"] == "MG") & (chunk["situacao_cadastral"] == "02")].copy()
        if selected.empty:
            continue
        selected["fonte_arquivo"] = zip_path.name
        filtered_chunks.append(selected)
        total_selected += len(selected)

estabelecimentos_mg = pd.concat(filtered_chunks, ignore_index=True) if filtered_chunks else pd.DataFrame(columns=ESTABELECIMENTOS_COLUMNS)

print(f"Linhas lidas: {total_rows:,}")
print(f"Estabelecimentos ativos em MG: {total_selected:,}")

estabelecimentos_mg.head()

## 6. Leitura das empresas relacionadas

Depois de filtrar os estabelecimentos, leio a tabela de empresas em partes e mantenho apenas os `cnpj_basico` que apareceram na base de MG. Isso reduz o volume antes da juncao.

In [ ]:
cnpj_basico_mg = set(estabelecimentos_mg["cnpj_basico"].dropna().unique())
empresas_files = find_zip_files("empresa")

empresa_chunks = []

for zip_path in empresas_files:
    print(f"Lendo {zip_path.name}...")
    for chunk in read_zip_csv(zip_path, EMPRESAS_COLUMNS, chunksize=CHUNKSIZE):
        selected = chunk[chunk["cnpj_basico"].isin(cnpj_basico_mg)].copy()
        if not selected.empty:
            empresa_chunks.append(selected)

empresas_mg = pd.concat(empresa_chunks, ignore_index=True) if empresa_chunks else pd.DataFrame(columns=EMPRESAS_COLUMNS)

print(f"Empresas relacionadas: {len(empresas_mg):,}")
empresas_mg.head()

## 7. Consolidacao da base bruta

Nesta etapa, junto estabelecimentos, empresas, municipios e CNAEs. Tambem monto o CNPJ completo e preparo as colunas finais definidas no notebook anterior.

In [ ]:
cnpj_mg = (
    estabelecimentos_mg
    .merge(empresas_mg, on="cnpj_basico", how="left")
    .merge(municipios, on="municipio_codigo", how="left")
    .merge(cnaes, on="cnae_fiscal_principal", how="left")
)

cnpj_mg["cnpj"] = build_cnpj(cnpj_mg)
cnpj_mg["opcao_simples"] = pd.NA
cnpj_mg["opcao_mei"] = pd.NA
cnpj_mg["data_extracao"] = pd.Timestamp.today().strftime("%Y-%m-%d")

cnpj_mg = cnpj_mg[OUTPUT_COLUMNS]

cnpj_mg.head()

## 8. Validacoes rapidas

Antes de salvar, verifico o volume extraido, a distribuicao por UF e situacao cadastral, alem de uma amostra dos municipios com mais registros.

In [ ]:
print(f"Total final de registros: {len(cnpj_mg):,}")
print(f"Total de CNPJs unicos: {cnpj_mg['cnpj'].nunique():,}")

display(cnpj_mg["uf"].value_counts(dropna=False))
display(cnpj_mg["situacao_cadastral"].value_counts(dropna=False))
display(cnpj_mg["municipio_nome"].value_counts(dropna=False).head(10))

## 9. Salvamento da extracao

A base final e salva no mesmo caminho criado no notebook `01`, substituindo o CSV vazio de cabecalhos pela extracao consolidada.

In [ ]:
cnpj_mg.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"Arquivo salvo em: {OUTPUT_CSV}")
print(f"Linhas salvas: {len(cnpj_mg):,}")

## 10. Leitura de conferencia

Por fim, leio uma amostra do CSV salvo para confirmar que o arquivo foi gravado corretamente.

In [ ]:
sample = pd.read_csv(OUTPUT_CSV, nrows=5, dtype=str)

print(sample.shape)
sample